<a href="https://colab.research.google.com/github/felicity8899/agent-trading/blob/main/Homework_1_Introduction_to_Machine_Learningipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [138]:
import yfinance as yf
import pandas as pd
import numpy as np

In [139]:
df = pd.read_html('https://www.iposcoop.com/ipos-recently-filed/')

In [140]:
df = df[0]

In [141]:
data = df[df['Expected To Trade'] == 'Withdrawn']
data.head(5)

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
17,2026-09-18,Rise Smart Group (Withdrawn),RSHL,American Trust Investment Services/Prime Numbe...,1.80,$4.00,$4.00,$7.20,Withdrawn,S/O
20,2026-09-17,C2 Capital Group (Withdrawn),CCLV,Dominari Securities,3.75,$4.00,$5.00,$16.88,Withdrawn,S/O
32,2026-09-15,OTSAW Ltd. (Withdrawn),OTSA,Aegis Capital,4.40,$4.50,$5.50,$22.00,Withdrawn,S/O
38,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.00,NaN,NaN,$100.00,Withdrawn,S/O
43,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.00,$4.00,$5.00,$9.00,Withdrawn,S/O


In [142]:
conditions = [
    df['Company'].str.contains('Technologies', case=False, na=False),
    df['Company'].str.contains('Acquisition Corp|Acquisition Corporation|Corp',case=False,na=False),
    df['Company'].str.contains('Inc|Incorporated', case=False, na=False),
    df['Company'].str.contains('Group', case=False, na=False),
    df['Company'].str.contains('Ltd|Limited', case=False, na=False),
    df['Company'].str.contains('Holdings|Holding', case=False, na=False),
]

choices = [
    'Technologies',
    'Acquisition Corp',
    'Inc.',
    'Group',
    'Limited',
    'Holdings',
]

df['Company Type'] = np.select(conditions, choices, default='Other')

In [143]:
parse = lambda x: pd.to_numeric(str(x).replace('$', '').strip(), errors='coerce')
df['Price Low'] = df['Price Low'].apply(parse).fillna(0)
df['Price High'] = df['Price High'].apply(parse).fillna(0)
df['Avg_price'] = (df['Price Low'] + df['Price High']) / 2


In [144]:
df['Shares_offered_value'] = df['Shares (millions)'].apply(parse).fillna(0) * df['Avg_price']

In [145]:
summary_df = df.groupby('Company Type')['Shares_offered_value'].sum().reset_index()
print(summary_df)

       Company Type  Shares_offered_value
0  Acquisition Corp            26068.0210
1             Group             2529.5950
2          Holdings             4999.5900
3              Inc.            21665.3257
4           Limited             1556.4069
5             Other            67616.1289
6      Technologies            75304.9450


In [146]:
df_list = pd.read_html('https://www.iposcoop.com/2025-pricings/')
end_date = pd.to_datetime('09-01-2025')
df2 = df_list[0]
df2['Offer Date'] = pd.to_datetime(df2['Offer Date'], errors='coerce')
df2['Return'] = df2['Return'].str.rstrip('%').astype('float') / 100.0
df2 = df2[(df2['Offer Date'] < end_date) & (df2['Return'] != 0)]
df2

,Company,Symbol,Industry,Offer Date,Shares (millions),Offer Price,1st Day Close,Current Price,Return,SCOOP Rating
68,"Picard Medical, Inc.",PMI,Health Care,2025-08-29,4.3,$4.00,$4.59,$1.68,-0.5800,S/O
69,GrowHub Limited (The),TGHL,Technology,2025-08-28,3.7,$4.00,$3.38,$0.40,-0.9000,S/O
70,TryHard Holdings Limited,THH,Consumer Services,2025-08-28,1.5,$4.00,$6.02,$29.47,6.3675,S/O
71,Curanex Pharmaceuticals,CURX,Health Care,2025-08-26,3.8,$4.00,$4.02,$0.37,-0.9075,S/O
72,"Cantor Equity Partners IV, Inc. (Stock-Only SPAC)",CEPF,Blank Check,2025-08-21,40.0,$10.00,$10.12,$10.12,0.0120,S/O
...,...,...,...,...,...,...,...,...,...,...
226,Mint Inc. Ltd.,MIMI,Other,2025-01-10,1.8,$4.00,$4.11,$5.30,0.3250,S/O
227,3 E Network Technology Group Limited,MASK,Technology,2025-01-08,1.3,$4.00,$3.25,$2.75,-0.3125,S/O
228,"Cantor Equity Partners I, Inc. (Stock-Only SPAC)",CEPO,Blank Check,2025-01-07,20.0,$10.00,$10.05,$10.10,0.0100,S/O
229,Zhengye Biotechnology Holding Limited,ZYBT,Health Care,2025-01-07,1.5,$4.00,$4.86,$4.22,0.0550,S/O


In [147]:
tickers = df2['Symbol'].dropna().unique().tolist()
stocks_df = yf.download(tickers, start='2024-01-01', end='2026-09-20', threads=True)

/tmp/ipykernel_25362/2316160296.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stocks_df = yf.download(tickers, start='2024-01-01', end='2026-09-20', threads=True)
[*********************100%***********************]  146 of 146 completed
ERROR:yfinance:
14 Failed downloads:
ERROR:yfinance:['SKBL', 'MJID', 'EPWK', 'EMPG', 'TBH', 'MTSR', 'PTNM', 'SDM', 'AGH', 'WGRX', 'CAEP', 'CEPT', 'AHL', 'MCTR']: YFTzMissingError('possibly delisted; no timezone found')


In [148]:
stocks_df = stocks_df.dropna(axis=1, how='all')

stocks_df.sort_index(ascending=True, inplace=True)

stocks_df.head(5)

Price      Close                                                  ... Volume  \
Ticker      AAPG AARD ADVB AII AIRO      ALM AMBQ ANPA ANTA APUS  ...   VOYG   
Date                                                              ...          
2024-01-02   NaN  NaN  NaN NaN  NaN  0.94725  NaN  NaN  NaN  NaN  ...    NaN   
2024-01-03   NaN  NaN  NaN NaN  NaN  0.93150  NaN  NaN  NaN  NaN  ...    NaN   
2024-01-04   NaN  NaN  NaN NaN  NaN  0.93375  NaN  NaN  NaN  NaN  ...    NaN   
2024-01-05   NaN  NaN  NaN NaN  NaN  0.93375  NaN  NaN  NaN  NaN  ...    NaN   
2024-01-08   NaN  NaN  NaN NaN  NaN  0.92250  NaN  NaN  NaN  NaN  ...    NaN   

Price                                                
Ticker     WETO WFF WTF WXM WYFI XHLD YMAT YMT ZYBT  
Date                                                 
2024-01-02  NaN NaN NaN NaN  NaN  NaN  NaN NaN  NaN  
2024-01-03  NaN NaN NaN NaN  NaN  NaN  NaN NaN  NaN  
2024-01-04  NaN NaN NaN NaN  NaN  NaN  NaN NaN  NaN  
2024-01-05  NaN NaN NaN NaN  NaN  NaN  NaN NaN  NaN  
2024-01-08  NaN NaN NaN NaN  NaN  NaN  NaN NaN  NaN  

[5 rows x 660 columns]

In [149]:
close_df = stocks_df['Close']
growth_df = (close_df / close_df.shift(252)) - 1
daily_returns = close_df.pct_change()
vol_df = daily_returns.rolling(30).std() * np.sqrt(252)

sharpe_df = (growth_df - 0.05) / vol_df


growth_multi = pd.concat({'growth_252d': growth_df}, axis=1)
vol_multi = pd.concat({'volatility': vol_df}, axis=1)
sharpe_multi = pd.concat({'Sharpe': sharpe_df}, axis=1)


stocks_df = pd.concat([stocks_df, growth_multi, vol_multi, sharpe_multi], axis=1)

/tmp/ipykernel_25362/3706121437.py:3: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  daily_returns = close_df.pct_change()


In [150]:
filtered_df = stocks_df.loc[:'2026-09-11']
filtered_df

Close                                                      \
Ticker           AAPG  AARD   ADVB        AII  AIRO        ALM       AMBQ   
Date                                                                        
2024-01-02        NaN   NaN    NaN        NaN   NaN   0.947250        NaN   
2024-01-03        NaN   NaN    NaN        NaN   NaN   0.931500        NaN   
2024-01-04        NaN   NaN    NaN        NaN   NaN   0.933750        NaN   
2024-01-05        NaN   NaN    NaN        NaN   NaN   0.933750        NaN   
2024-01-08        NaN   NaN    NaN        NaN   NaN   0.922500        NaN   
...               ...   ...    ...        ...   ...        ...        ...   
2026-09-04  17.615000  5.38  9.335  26.000000  7.40  17.570000  57.450001   
2026-09-08  17.440001  5.33  8.980  26.200001  7.33  19.120001  59.619999   
2026-09-09  17.309999  4.92  8.500  25.920000  6.88  18.330000  63.770000   
2026-09-10  16.670000  5.01  8.850  25.410000  6.94  16.570000  60.660000   
2026-09-11  16.100000  5.13  8.190  25.930000  7.25  15.490000  63.520000   

                                ...    Sharpe                                \
Ticker      ANPA   ANTA   APUS  ...      VOYG      WETO       WFF       WTF   
Date                            ...                                           
2024-01-02   NaN    NaN    NaN  ...       NaN       NaN       NaN       NaN   
2024-01-03   NaN    NaN    NaN  ...       NaN       NaN       NaN       NaN   
2024-01-04   NaN    NaN    NaN  ...       NaN       NaN       NaN       NaN   
2024-01-05   NaN    NaN    NaN  ...       NaN       NaN       NaN       NaN   
2024-01-08   NaN    NaN    NaN  ...       NaN       NaN       NaN       NaN   
...          ...    ...    ...  ...       ...       ...       ...       ...   
2026-09-04  3.97  3.760  2.420  ...  0.177168 -0.105741 -0.335256 -0.620753   
2026-09-08  3.30  3.445  2.300  ...  0.152815 -0.107642 -0.341264 -0.646620   
2026-09-09  3.11  3.572  2.290  ...  0.095767 -0.107670 -0.357396 -0.841688   
2026-09-10  2.77  3.890  2.170  ...  0.108834 -0.107871 -0.362641 -0.949715   
2026-09-11  2.62  3.725  2.105  ...  0.149979 -0.108214 -0.345893 -0.993553   

                                                                        
Ticker           WXM      WYFI      XHLD      YMAT       YMT      ZYBT  
Date                                                                    
2024-01-02       NaN       NaN       NaN       NaN       NaN       NaN  
2024-01-03       NaN       NaN       NaN       NaN       NaN       NaN  
2024-01-04       NaN       NaN       NaN       NaN       NaN       NaN  
2024-01-05       NaN       NaN       NaN       NaN       NaN       NaN  
2024-01-08       NaN       NaN       NaN       NaN       NaN       NaN  
...              ...       ...       ...       ...       ...       ...  
2026-09-04 -0.274463  0.105947  0.123476 -1.299395 -0.693410 -0.309950  
2026-09-08 -0.275821  0.145074  0.144179 -0.962146 -0.714064 -0.316982  
2026-09-09 -0.277184  0.108211  0.177327 -0.303157 -0.713888 -0.320982  
2026-09-10 -0.284083 -0.030103  0.154193 -0.326724 -0.709735 -0.334200  
2026-09-11 -0.289666 -0.207703  0.183732 -0.332692 -0.710717 -0.340384  

[676 rows x 1056 columns]

In [151]:
features={}

for m in range(1,13):
  days  = m * 21
  features[f'future_growth_{m}_m'] = stocks_df['Close']/stocks_df['Close'].shift(-days)

features = pd.concat(features, axis=1)
growth_full = pd.concat([growth_multi, features,sharpe_multi], axis=1)
growth_full

growth_252d                                                    \
Ticker            AAPG      AARD      ADVB       AII      AIRO       ALM   
Date                                                                       
2024-01-02         NaN       NaN       NaN       NaN       NaN       NaN   
2024-01-03         NaN       NaN       NaN       NaN       NaN       NaN   
2024-01-04         NaN       NaN       NaN       NaN       NaN       NaN   
2024-01-05         NaN       NaN       NaN       NaN       NaN       NaN   
2024-01-08         NaN       NaN       NaN       NaN       NaN       NaN   
...                ...       ...       ...       ...       ...       ...   
2026-09-14   -0.588619 -0.287548 -0.202869  0.394797 -0.634945  2.237885   
2026-09-15   -0.604548 -0.415600 -0.220077  0.389785 -0.639027  2.147982   
2026-09-16   -0.601346 -0.482412 -0.257692  0.362472 -0.644059  1.800416   
2026-09-17   -0.608008 -0.461014 -0.348754  0.378599 -0.627837  1.901437   
2026-09-18   -0.613350 -0.509572 -0.419183  0.321259 -0.633700  1.836066   

                                                    ...    Sharpe            \
Ticker          AMBQ      ANPA      ANTA      APUS  ...      VOYG      WETO   
Date                                                ...                       
2024-01-02       NaN       NaN       NaN       NaN  ...       NaN       NaN   
2024-01-03       NaN       NaN       NaN       NaN  ...       NaN       NaN   
2024-01-04       NaN       NaN       NaN       NaN  ...       NaN       NaN   
2024-01-05       NaN       NaN       NaN       NaN  ...       NaN       NaN   
2024-01-08       NaN       NaN       NaN       NaN  ...       NaN       NaN   
...              ...       ...       ...       ...  ...       ...       ...   
2026-09-14  0.630025 -0.946292 -0.694491 -0.892268  ...  0.056760 -0.134633   
2026-09-15  0.766328 -0.940509 -0.705290 -0.921154  ...  0.134614 -0.135386   
2026-09-16  0.726723 -0.943038 -0.709106 -0.920792  ...  0.169717 -0.135867   
2026-09-17  0.797866 -0.950149 -0.715514 -0.912105  ...  0.347478 -0.135997   
2026-09-18  0.985441 -0.948750 -0.719492 -0.901117  ...  0.339206 -0.136077   

                                                                        \
Ticker           WFF       WTF       WXM      WYFI      XHLD      YMAT   
Date                                                                     
2024-01-02       NaN       NaN       NaN       NaN       NaN       NaN   
2024-01-03       NaN       NaN       NaN       NaN       NaN       NaN   
2024-01-04       NaN       NaN       NaN       NaN       NaN       NaN   
2024-01-05       NaN       NaN       NaN       NaN       NaN       NaN   
2024-01-08       NaN       NaN       NaN       NaN       NaN       NaN   
...              ...       ...       ...       ...       ...       ...   
2026-09-14 -0.403657 -1.316926 -0.291413 -0.171349  0.097059 -0.341996   
2026-09-15 -0.382422 -1.258628 -0.292184 -0.188205  0.100749 -0.349402   
2026-09-16 -0.362184 -1.112836 -0.292459 -0.169666  0.115084 -0.357652   
2026-09-17 -0.335429 -1.085825 -0.292621 -0.111646  0.105677 -0.350060   
2026-09-18 -0.331672 -1.092958 -0.296194 -0.129094  0.292225 -0.347468   

                                
Ticker           YMT      ZYBT  
Date                            
2024-01-02       NaN       NaN  
2024-01-03       NaN       NaN  
2024-01-04       NaN       NaN  
2024-01-05       NaN       NaN  
2024-01-08       NaN       NaN  
...              ...       ...  
2026-09-14 -0.714922 -0.336751  
2026-09-15 -0.739601 -0.338753  
2026-09-16 -0.806157 -0.346665  
2026-09-17 -0.807172 -0.990506  
2026-09-18 -0.849179 -1.035987  

[681 rows x 1848 columns]

In [152]:
close_long = stocks_df['Close'].unstack().reset_index()
close_long.columns = ['Ticker', 'Date', 'Close']

entry_df = (close_long.dropna(subset=['Close'])
            .sort_values(by=['Ticker', 'Date'])
            .groupby('Ticker')
            .first()
            .reset_index())

entry_df.rename(columns={'Date': 'min_date', 'Close': 'Entry_Price'}, inplace=True)
growth_full_long = growth_full.stack().reset_index()
full_df = pd.merge(
    entry_df,
    growth_full_long,
    left_on=['Ticker', 'min_date'],
    right_on=['Ticker', 'Date'],
    how='inner')

filter_df = full_df[full_df['min_date']< end_date]
ticker_list = filter_df['Ticker'].tolist()
ticker_list
sharpe_matrix = filtered_df['Sharpe']
final_df = sharpe_matrix[ticker_list]
sharpe_on_that_day = final_df.loc[pd.to_datetime('2026-09-11')]

final_single_number = sharpe_on_that_day.median()

final_single_number

/tmp/ipykernel_25362/2004956438.py:11: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  growth_full_long = growth_full.stack().reset_index()


-0.3871696711261834

In [153]:
full_df.describe()

,min_date,Entry_Price,Date,growth_252d,future_growth_1_m,future_growth_2_m,future_growth_3_m,future_growth_4_m,future_growth_5_m,future_growth_6_m,future_growth_7_m,future_growth_8_m,future_growth_9_m,future_growth_10_m,future_growth_11_m,future_growth_12_m,Sharpe
count,132,132.000000,132,0.0,132.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,130.000000,130.000000,0.0
mean,2025-04-22 07:38:10.909090816,193.113739,2025-04-22 07:38:10.909090816,NaN,1.480264,1.626312,2.108704,2.494470,2.858471,3.212049,3.925697,4.440639,5.286707,6.363997,8.288176,10.374572,NaN
min,2024-01-02 00:00:00,0.600000,2024-01-02 00:00:00,NaN,0.113800,0.062240,0.061183,0.087822,0.122208,0.101066,0.048213,0.046793,0.036196,0.028829,0.051031,0.045363,NaN
25%,2025-02-18 18:00:00,4.577500,2025-02-18 18:00:00,NaN,0.910716,0.870449,0.910517,0.850129,0.929014,0.960492,0.982281,0.969330,0.978219,0.973513,0.934391,0.960117,NaN
50%,2025-04-21 00:00:00,17.408814,2025-04-21 00:00:00,NaN,1.069145,1.123077,1.208955,1.348294,1.426948,1.349945,1.483803,1.654371,1.699095,1.873922,2.053077,1.929061,NaN
75%,2025-07-23 00:00:00,37.365000,2025-07-23 00:00:00,NaN,1.410765,1.752191,2.274799,2.644762,2.557043,3.247445,4.256332,5.097304,5.728952,5.741997,6.479158,6.377280,NaN
max,2026-08-04 00:00:00,14240.000000,2026-08-04 00:00:00,NaN,13.709677,9.416446,11.092986,15.642857,31.012819,45.299623,75.593747,66.092893,68.140844,83.702417,115.190479,206.896558,NaN
std,NaN,1293.778647,NaN,NaN,1.542358,1.544297,2.167713,3.055052,4.157613,5.316905,7.766592,7.666390,9.317538,13.008530,18.131390,29.373351,NaN


In [160]:
full_df.sort_values(by='min_date',ascending=False, inplace=True)
full_df

,Ticker,min_date,Entry_Price,Date,growth_252d,future_growth_1_m,future_growth_2_m,future_growth_3_m,future_growth_4_m,future_growth_5_m,future_growth_6_m,future_growth_7_m,future_growth_8_m,future_growth_9_m,future_growth_10_m,future_growth_11_m,future_growth_12_m,Sharpe
27,CIGL,2026-08-04,0.630000,2026-08-04,NaN,1.536585,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
117,TOPW,2025-09-04,32.483055,2025-09-04,NaN,0.914398,0.867363,0.945576,1.000205,1.126905,1.110722,1.137014,0.965843,1.000763,1.027535,0.972192,0.985453,NaN
98,PMI,2025-08-29,229.500000,2025-08-29,NaN,0.540000,1.399390,2.154930,2.622857,2.622857,4.781250,4.371429,10.455581,17.000000,27.159764,44.736841,59.455960,NaN
112,TGHL,2025-08-28,3.380000,2025-08-28,NaN,2.036145,4.246231,6.329588,9.209809,5.121212,8.387097,12.071429,9.234973,2.503704,6.003552,2.991151,4.441525,NaN
113,THH,2025-08-27,241.899994,2025-08-27,NaN,3.322802,2.717977,2.314832,1.428403,31.012819,45.299623,75.593747,66.092893,68.140844,83.702417,115.190479,118.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,PPCB,2024-01-02,0.600000,2024-01-02,NaN,2.033898,4.137931,5.714286,15.000001,6.666667,10.909091,17.142858,24.000001,40.000002,40.000002,40.000002,30.000002,NaN
28,CIIT,2024-01-02,57.959999,2024-01-02,NaN,1.037594,1.037594,0.591429,0.636923,0.752727,0.394286,0.153903,0.192558,0.205459,0.206484,0.207000,0.207000,NaN
21,CAPS,2024-01-02,5.200000,2024-01-02,NaN,1.000000,1.000000,1.000000,1.000000,1.268293,1.268293,1.155556,1.155556,1.600000,1.625000,1.625000,1.890909,NaN
13,AVBH,2024-01-02,20.350000,2024-01-02,NaN,1.017500,1.079576,1.065445,1.085333,1.072181,1.108992,1.094086,1.035623,1.043590,0.992683,0.920398,0.933486,NaN


In [155]:
import gdown
file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)
df = pd.read_parquet("data.parquet", engine="pyarrow")

Downloading...
From (original): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-
From (redirected): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-&confirm=t&uuid=069d4a0d-a7df-4850-bb0f-5fbccd65b799
To: /content/data.parquet
100%|██████████| 130M/130M [00:01<00:00, 98.2MB/s]


In [156]:
df['Date'] = pd.to_datetime(df['Date'])

selected_df = df[
    (df['Date'] >= '2000-01-01') &
    (df['Date'] <= '2025-06-01') &
    (df['rsi'] < 30)
]

net_income = 1000 * (selected_df['growth_future_30d'] - 1).sum()

profit_in_k = net_income / 1000

print(f"select : {len(selected_df)}")
print(f"Net income earned (in $ thousands): {profit_in_k:.0f}")

select : 5206
Net income earned (in $ thousands): 66
